In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Check Dataset 

In [ ]:
df_train = pd.read_csv("/kaggle/input/titanic/train.csv")
df_test = pd.read_csv("/kaggle/input/titanic/test.csv")
df_test_ = pd.read_csv("/kaggle/input/titanic/test.csv")
test_y = pd.read_csv("/kaggle/input/titanic/gender_submission.csv")



In [ ]:
df_train.head()

In [ ]:
df_test.head()

In [ ]:
df_train.info()

In [ ]:
df_test.info()

In [ ]:
test_y.describe()

**Removed columns** : PassengerId, Ticket, Embarked

**Y - Survived**


**Columns for change**

* Pclass > change to P1, P2 -> both 0 means passanger form 3rd class
* Sex > 1-feemale 0 male
* Age , add missing values and normalize
* Fare normalize
  
**Try feature engineering**
* family size, is alone
* title (from Name)
* Sex * Pclass ?

In [ ]:
def check_P1(number):
    if number == 1:
        return 1
    else:
        return 0

def check_P2(number):
    if number == 2:
        return 1
    else:
        return 0

def check_Sex(sex):
    if sex == 'female':
        return 1
    else:
        return 0

In [ ]:
df_train['P1'] = df_train['Pclass'].apply(check_P1)
df_train['P2'] = df_train['Pclass'].apply(check_P2)
df_train['Sex'] = df_train['Sex'].apply(check_Sex)

In [ ]:
df_test['P1'] = df_test['Pclass'].apply(check_P1)
df_test['P2'] = df_test['Pclass'].apply(check_P2)
df_test['Sex'] = df_test['Sex'].apply(check_Sex)

In [ ]:
df_train['Age'].fillna(df_train['Age'].median(), inplace=True)
df_train['Age'] = (df_train['Age'] - df_train['Age'].min()) / (df_train['Age'].max() - df_train['Age'].min())
df_train['Fare'] = (df_train['Fare'] - df_train['Fare'].min()) / (df_train['Fare'].max() - df_train['Fare'].min())
df_test['Age'].fillna(df_test['Age'].median(), inplace=True)
df_test['Fare'].fillna(df_test['Fare'].median(), inplace=True)
df_test['Age'] = (df_test['Age'] - df_test['Age'].min()) / (df_test['Age'].max() - df_test['Age'].min())
df_test['Fare'] = (df_test['Fare'] - df_test['Fare'].min()) / (df_test['Fare'].max() - df_test['Fare'].min())

In [ ]:
df_train['FamilySize'] = df_train['SibSp'] + df_train['Parch']+1
df_test['FamilySize'] = df_test['SibSp'] + df_test['Parch']+1

In [ ]:
df_train['FamilySize'] = (df_train['FamilySize'] - df_train['FamilySize'].min()) / (df_train['FamilySize'].max() - df_train['FamilySize'].min())
df_test['FamilySize'] = (df_test['FamilySize'] - df_test['FamilySize'].min()) / (df_test['FamilySize'].max() - df_test['FamilySize'].min())

In [ ]:
df_train['IsAlone'] = (df_train['FamilySize'] == 0).astype(int)
df_test['IsAlone'] = (df_test['FamilySize'] == 0).astype(int)

In [ ]:
df_train.head()

In [ ]:
train_y = df_train['Survived']
df_train = df_train.drop(['PassengerId', 'Name', 'Ticket',  'Embarked', 'Pclass', 'Survived', 'Cabin', 'SibSp', 'Parch'], axis=1)

In [ ]:
df_train.head()

In [ ]:
df_train.describe()

In [ ]:
df_test = df_test.drop(['PassengerId', 'Name', 'Ticket',  'Embarked', 'Pclass', 'Cabin', 'SibSp', 'Parch'], axis=1)

In [ ]:
df_test.head()

In [ ]:
df_test.describe()

In [ ]:
print(df_train.isna().sum())

In [ ]:
# Set random seed

torch.manual_seed(0)

In [ ]:
# Create the data class

class Data(Dataset):

    # Constructor
    def __init__(self, df, Y):
        self.x = torch.tensor(df.values, dtype=torch.float32)
        self.y = torch.tensor(Y.values, dtype=torch.float32).view(-1, 1)
        self.len = self.x.shape[0]

    # Getter
    def __getitem__(self, index):
        return self.x[index], self.y[index]

    # Get length
    def __len__(self):
        return self.len

In [ ]:
train_data = Data(df_train,train_y )
test_data = Data(df_test, test_y['Survived'])

In [ ]:
train_data.len

In [ ]:
# Create logistic_regression class

class logistic_regression(nn.Module):

    # Constructor
    def __init__(self, n_inputs):
        super(logistic_regression, self).__init__()
        self.linear = nn.Linear(n_inputs, 1)

    # Prediction
    def forward(self, x):
        yhat = torch.sigmoid(self.linear(x))
        return yhat

In [ ]:
model = logistic_regression(7) #7 features

In [ ]:
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False)

In [ ]:
# Build in criterion
criterion = nn.BCELoss()
learning_rate = 2
optimizer = torch.optim.SGD(model.parameters(), lr = learning_rate)

In [ ]:
def evaluate(loader, model):
    model.eval()
    correct, total, total_loss = 0, 0, 0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            yhat = model(x_batch)
            loss = criterion(yhat, y_batch)
            total_loss += loss.item() * y_batch.size(0)
            predicted = (yhat >= 0.5).float()
            correct += (predicted == y_batch).sum().item()
            total += y_batch.size(0)
    return total_loss / total, correct / total

In [ ]:
epochs = 30
train_losses, test_losses = [], []
train_accs, test_accs = [], []

for epoch in range(epochs):
    model.train()
    for x_batch, y_batch in train_loader:
        yhat = model(x_batch)
        #print(yhat.min().item(), yhat.max().item())
        loss = criterion(yhat, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()


    train_loss, train_acc = evaluate(train_loader, model)
    test_loss, test_acc = evaluate(test_loader, model)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    train_accs.append(train_acc)
    test_accs.append(test_acc)
    if epoch % 15 == 0:
        print(f"Epoch {epoch+1:02d}: "
          f"Train loss={train_loss:.4f}, acc={train_acc:.3f} | "
          f"Test loss={test_loss:.4f}, acc={test_acc:.3f}")
    

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(train_losses, label='Train loss')
plt.plot(test_losses, label='Test loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over time')
plt.legend()

plt.subplot(1,2,2)
plt.plot(train_accs, label='Train acc')
plt.plot(test_accs, label='Test acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over time')
plt.legend()

plt.show()

In [ ]:
X_submisstion_tensor = torch.tensor(df_test.values, dtype=torch.float32)

In [ ]:
model.eval()  # no more training mode
with torch.no_grad():
    y_pred = model(X_submisstion_tensor)
    y_class = (y_pred >= 0.5).float()  

In [ ]:
df_test_['Survived'] = y_class.numpy().astype(int)

In [ ]:
df_test.head()

In [ ]:
gender_submission = df_test_[['PassengerId', 'Survived']]

In [ ]:
gender_submission.to_csv("submission.csv", index=False)